# ICICLE Inference from SMILES
Given a filtered list of molecules, this notebook:
1. Loads an ICICLE checkpoint and predicts EIMS spectra
2. Loads experimental spectra from NIST HDF5 (when available)
3. Displays per-molecule:
    - Dataset split label
    - Predicted spectrum
    - Mirror plot (pred vs exp) + similarity scores
    - Interactive spectrum with fragment hover
    - Fragmentation DAG

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import base64
import io

import h5py
import ipywidgets as widgets
import matplotlib.cm as mcm
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import HTML, display
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw

from icicle.analysis.metrics import (
    composite_weighted_cosine_similarity,
    cosine_similarity,
    entropy_similarity,
    weighted_cosine_similarity,
)
from icicle.models.eims_predictor import EIMSPredictorFromFullEnumeration
from icicle.utils.visualization import set_style
from icicle.utils.visualization.mass_spectra import (
    plot_mass_spectrum,
    plot_mirrored_spectra,
)

set_style("manuscript")

## Configuration

In [ ]:
CKPT = "/home/magled/icicle-dev/checkpoints/06-20-50_icicle_rdm_no_xeno_best_so_far/checkpoints/best-model-val_loss=0.1473-epoch=98.ckpt"

METADATA_PATH = "/home/magled/icicle-dev/data/NIST2023_GCMS_main/metadata.tsv"
SPECTRA_PATH = "/home/magled/icicle-dev/data/NIST2023_GCMS_main/spectra.hdf5"
SPLIT_PATH = "/home/magled/icicle-dev/data/NIST2023_GCMS_main/splits/random_no_xeno_aas_deduplicated.tsv"

DEVICE = "cuda:0"
MAX_NODES = 50
THRESHOLD = 0.01

## Build Molecule List

Edit the filter below to select any subset. The notebook works with any `df_filtered` that has columns: `mol_id`, `inchi_key`, `standardized_smiles`, `split`.

In [ ]:
metadata = pd.read_csv(METADATA_PATH, sep="\t")
split_df = pd.read_csv(
    SPLIT_PATH, sep="\t"
)  # columns: mol_id, inchi_key, split

# ── edit filter here ──────────────────────────────────────────────────────────
mask = (
    metadata["standardized_smiles"].str.contains("Br", na=False)
    & metadata["mw"].between(200, 250)
    & metadata["retention_index"].notna()
    & metadata["has_spectrum"]
)
df_filtered = metadata[mask].copy()

# attach split label
df_filtered = df_filtered.merge(
    split_df[["inchi_key", "split"]], on="inchi_key", how="left"
)

# restrict to test split; remove to see all
df_filtered = df_filtered[df_filtered["split"] == "test"]

# further filter: non-aromatic only (for illustration)
df_filtered = df_filtered[
    ~df_filtered["standardized_smiles"].str.contains("c", na=False)
]

df_filtered = df_filtered.reset_index(drop=True)
print(f"{len(df_filtered)} molecules selected")
df_filtered[
    [
        "mol_id",
        "inchi_key",
        "standardized_smiles",
        "mw",
        "split",
        "retention_index",
    ]
].head(10)

## Load Model

In [ ]:
eims_predictor = EIMSPredictorFromFullEnumeration(
    min_mz=0, max_mz=750, bin_width=1.0
)
eims_predictor.load_from_checkpoint(CKPT)
print("Model loaded.")

## Helper Functions

In [ ]:
def load_exp_spectrum(mol_id: str, n_bins: int = 750) -> np.ndarray | None:
    """Load and bin experimental spectrum from NIST HDF5. Returns None if missing."""
    with h5py.File(SPECTRA_PATH, "r") as f:
        if str(mol_id) not in f:
            return None
        masses = f[str(mol_id)]["masses"][:]
        intensities = f[str(mol_id)]["intensities"][:]
    spectrum = np.zeros(n_bins)
    for mz, inten in zip(masses, intensities):
        idx = int(mz)
        if idx < n_bins:
            spectrum[idx] = inten
    if spectrum.max() > 0:
        spectrum /= spectrum.max()
    return spectrum


def compute_similarities(pred: np.ndarray, exp: np.ndarray) -> dict:
    """Compute cosine, entropy, weighted cosine, and composite similarities."""
    mz = np.arange(len(pred))
    return {
        "cosine": cosine_similarity(pred, exp),
        "entropy": entropy_similarity(pred, exp),
        "weighted_cosine": weighted_cosine_similarity(
            pred, exp, mz_values=mz, weighting_scheme="nist_gc"
        ),
        "composite": composite_weighted_cosine_similarity(
            pred, exp, mz_values=mz, weighting_scheme="nist_gc"
        ),
    }

In [ ]:
H_MASS = 1.00794


def interactive_spectrum(
    result: dict,
    smiles: str | None = None,
    title: str | None = None,
    height: int = 450,
    max_h_shift: int = 6,
) -> None:
    """Interactive Plotly spectrum: click peak or use dropdown to inspect fragment structures."""
    mz_bins = result["mz_bins"]
    intensities = result["intensities"] / np.max(result["intensities"])
    fragments = result.get("fragments", {})

    frag_list = []
    for frag_mz, frag_info in fragments.items():
        base_bin = int(np.abs(mz_bins - frag_mz).argmin())
        frag_list.append((frag_mz, base_bin, frag_info))

    frag_base_masses = (
        np.array([f[0] for f in frag_list]) if frag_list else np.array([])
    )

    peak_mask = intensities > 0.005
    mz_plot = mz_bins[peak_mask]
    int_plot = intensities[peak_mask]
    peak_indices = np.where(peak_mask)[0]
    n_bars = len(mz_plot)

    bin_to_frag_idx: dict = {}
    frag_idx_to_bars: dict = {i: [] for i in range(len(frag_list))}
    if len(frag_base_masses) > 0:
        for bar_idx, bin_idx in enumerate(peak_indices):
            peak_mz = mz_bins[bin_idx]
            distances = np.abs(frag_base_masses - peak_mz)
            nearest_idx = int(np.argmin(distances))
            if distances[nearest_idx] <= max_h_shift * H_MASS + 0.5:
                bin_to_frag_idx[bin_idx] = nearest_idx
                frag_idx_to_bars[nearest_idx].append(bar_idx)

    bin_to_bar = {
        bin_idx: bar_idx for bar_idx, bin_idx in enumerate(peak_indices)
    }
    bar_to_bin = {bar_idx: bin_idx for bin_idx, bar_idx in bin_to_bar.items()}

    default_color = "rgba(31, 119, 180, 0.7)"
    faded_color = "rgba(31, 119, 180, 0.2)"
    highlight_color = "rgba(220, 50, 50, 0.9)"
    hl_hshift_color = "rgba(220, 100, 100, 0.6)"

    fig = go.FigureWidget()
    fig.add_trace(
        go.Bar(
            x=mz_plot,
            y=int_plot,
            width=0.8,
            marker_color=[default_color] * n_bars,
            hovertemplate="m/z: %{x:.1f}<br>Intensity: %{y:.3f}<extra></extra>",
        )
    )

    if smiles:
        mol = Chem.MolFromSmiles(smiles)
        if mol:
            img = Draw.MolToImage(mol, size=(200, 200))
            buf = io.BytesIO()
            img.save(buf, format="PNG")
            b64 = base64.b64encode(buf.getvalue()).decode()
            fig.add_layout_image(
                source=f"data:image/png;base64,{b64}",
                xref="paper",
                yref="paper",
                x=0.02,
                y=0.98,
                sizex=0.25,
                sizey=0.25,
                xanchor="left",
                yanchor="top",
            )
            x_max = Descriptors.ExactMolWt(mol) + 10
        else:
            x_max = float(mz_plot[-1]) + 10
    else:
        x_max = float(mz_plot[-1]) + 10

    fig.update_layout(
        title=title or "",
        xaxis_title="m/z",
        yaxis_title="Relative Intensity",
        xaxis=dict(range=[0, x_max]),
        yaxis=dict(range=[0, 1.15]),
        height=height,
        plot_bgcolor="white",
        showlegend=False,
    )
    fig.update_xaxes(showgrid=False, showline=True, linecolor="black")
    fig.update_yaxes(showgrid=False, showline=True, linecolor="black")

    if not fragments:
        display(fig)
        return

    sorted_frag_indices = sorted(
        range(len(frag_list)),
        key=lambda i: intensities[frag_list[i][1]],
        reverse=True,
    )
    options = [("Select a fragment...", -1)]
    for fi in sorted_frag_indices:
        frag_mz, base_bin, frag_info = frag_list[fi]
        inten = intensities[base_bin]
        n_peaks = len(frag_idx_to_bars[fi])
        h_label = f", {n_peaks} peaks" if n_peaks > 1 else ""
        options.append(
            (
                f"m/z {frag_mz:.1f}  —  {frag_info.get('form', '?')}  (I={inten:.3f}{h_label})",
                fi,
            )
        )

    dropdown = widgets.Dropdown(
        options=options,
        value=-1,
        description="Fragment:",
        style={"description_width": "80px"},
        layout=widgets.Layout(width="450px"),
    )
    frag_output = widgets.Output(layout=widgets.Layout(min_height="60px"))

    def _highlight(frag_idx: int) -> None:
        frag_output.clear_output(wait=True)
        if frag_idx < 0:
            with fig.batch_update():
                fig.data[0].marker.color = [default_color] * n_bars
            return
        frag_mz, base_bin, frag_info = frag_list[frag_idx]
        related_bars = frag_idx_to_bars[frag_idx]
        base_bar = bin_to_bar.get(base_bin)
        colors = [faded_color] * n_bars
        for bi in related_bars:
            colors[bi] = hl_hshift_color
        if base_bar is not None and base_bar in related_bars:
            colors[base_bar] = highlight_color
        with fig.batch_update():
            fig.data[0].marker.color = colors
        with frag_output:
            mol = frag_info["structure"]
            highlights = frag_info.get("highlights", {})
            img = Draw.MolToImage(
                mol,
                highlightAtoms=highlights.get("atoms", []),
                highlightBonds=highlights.get("bonds", []),
                size=(350, 350),
            )
            buf = io.BytesIO()
            img.save(buf, format="PNG")
            b64 = base64.b64encode(buf.getvalue()).decode()
            shifted = [f"{mz_plot[bi]:.0f}" for bi in related_bars]
            h_shift_info = (
                f"<br><b>All peaks:</b> m/z {', '.join(shifted)}"
                if len(related_bars) > 1
                else ""
            )
            display(
                HTML(
                    f"<div style='display:flex;align-items:center;gap:20px;'>"
                    f"<img src='data:image/png;base64,{b64}' width='250'/>"
                    f"<div><b>m/z:</b> {frag_mz:.1f}<br>"
                    f"<b>Intensity:</b> {intensities[base_bin]:.3f}<br>"
                    f"<b>Formula:</b> {frag_info.get('form', '?')}"
                    f"{h_shift_info}</div></div>"
                )
            )

    def on_select(change: dict) -> None:
        _highlight(change["new"])

    def on_click(trace, points, selector) -> None:
        if not points.point_inds:
            return
        bar_idx = points.point_inds[0]
        bin_idx = bar_to_bin.get(bar_idx)
        if bin_idx is not None and bin_idx in bin_to_frag_idx:
            dropdown.value = bin_to_frag_idx[bin_idx]

    dropdown.observe(on_select, names="value")
    fig.data[0].on_click(on_click)
    hint = widgets.HTML(
        "<span style='color:#888;font-size:12px;'>Click a peak or use the dropdown</span>"
    )
    display(widgets.VBox([fig, widgets.HBox([dropdown, hint]), frag_output]))

In [ ]:
def plot_prediction_dag(
    result: dict, title: str | None = None, cmap_name: str = "Blues"
) -> plt.Figure | None:
    """Plot fragmentation DAG. Node border color encodes relative intensity."""
    dag_nodes = result.get("dag_nodes", {})
    if not dag_nodes:
        print("No DAG nodes in result.")
        return None

    smiles = result.get("smiles", "")
    intensities = result["intensities"]
    mz_bins = result["mz_bins"]
    max_inten = intensities.max() if intensities.max() > 0 else 1.0
    cmap = mcm.get_cmap(cmap_name)

    for node in dag_nodes.values():
        bin_idx = int(np.abs(mz_bins - node["base_mass"]).argmin())
        node["rel_inten"] = float(intensities[bin_idx]) / max_inten

    roots = [
        h
        for h, n in dag_nodes.items()
        if n["tree_depth"] == 0 and not n["parent_hashes"]
    ]
    root_hash = (
        max(roots, key=lambda h: dag_nodes[h]["base_mass"]) if roots else None
    )

    depth_groups: dict = {}
    for h, n in dag_nodes.items():
        depth_groups.setdefault(n["tree_depth"], []).append(h)

    y_spacing, x_spacing = 5.0, 5.0
    pos: dict = {}
    for depth, hashes in depth_groups.items():
        y = -depth * y_spacing
        n = len(hashes)
        xs = [i * x_spacing - (n - 1) * x_spacing / 2 for i in range(n)]
        for h, x in zip(hashes, xs):
            pos[h] = (x, y)

    edges = [
        (ph, h)
        for h, node in dag_nodes.items()
        for ph in node["parent_hashes"]
        if ph in dag_nodes
    ]

    all_x = [p[0] for p in pos.values()]
    all_y = [p[1] for p in pos.values()]
    pad = 2
    fw = max((max(all_x) - min(all_x)) / 2.5 + pad + 1.5, 5)
    fh = max((max(all_y) - min(all_y)) / 1.0 + pad, 6)

    fig, ax = plt.subplots(figsize=(fw, fh), dpi=150)
    ax.set_xlim(min(all_x) - pad, max(all_x) + pad)
    ax.set_ylim(min(all_y) - pad, max(all_y) + pad)
    ax.axis("off")
    ax.set_title(
        title or f"Fragmentation DAG\n{smiles}", fontsize=12, fontweight="bold"
    )

    for ph, ch in edges:
        px, py = pos[ph]
        cx, cy = pos[ch]
        ax.annotate(
            "",
            xy=(cx, cy),
            xytext=(px, py),
            arrowprops=dict(arrowstyle="->", lw=1.5, color="gray", alpha=0.5),
        )

    for h, node in dag_nodes.items():
        x, y = pos[h]
        mol = node["structure"]
        highlights = node.get("highlights", {})
        try:
            img = Draw.MolToImage(
                mol,
                highlightAtoms=highlights.get("atoms", []),
                highlightColors={
                    a: (1.0, 0.7, 0.7) for a in highlights.get("atoms", [])
                },
                size=(300, 300),
            )
        except Exception:
            img = Draw.MolToImage(mol, size=(300, 300))

        rgba = cmap(0.15 + 0.85 * node["rel_inten"])
        lw = 4 if h == root_hash else 2.5
        im = OffsetImage(np.array(img), zoom=0.35, alpha=0.95)
        ab = AnnotationBbox(
            im, (x, y), frameon=True, boxcoords="data", pad=0.2
        )
        ab.patch.set_edgecolor(rgba)
        ab.patch.set_linewidth(lw)
        ab.patch.set_facecolor("white")
        ax.add_artist(ab)

        if h == root_hash:
            ab2 = AnnotationBbox(
                OffsetImage(np.array(img), zoom=0.35, alpha=0),
                (x, y),
                frameon=True,
                boxcoords="data",
                pad=0.2,
            )
            ab2.patch.set_edgecolor("orange")
            ab2.patch.set_linewidth(6)
            ab2.patch.set_facecolor("none")
            ax.add_artist(ab2)

        pct = node["rel_inten"] * 100
        label = f"{node['form']}\nm/z {node['base_mass']:.1f}  {pct:.0f}%\ndepth {node['tree_depth']}"
        ax.text(
            x,
            y - 3,
            label,
            fontsize=7,
            ha="center",
            va="top",
            color=mcolors.to_hex(rgba),
            bbox=dict(
                boxstyle="round,pad=0.3",
                facecolor="white",
                edgecolor=rgba,
                alpha=0.9,
            ),
        )

    sm = mcm.ScalarMappable(
        cmap=cmap, norm=mcolors.Normalize(vmin=0, vmax=100)
    )
    sm.set_array([])
    cbar = fig.colorbar(sm, ax=ax, fraction=0.02, pad=0.02, aspect=30)
    cbar.set_label("Relative Intensity (%)", fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    from matplotlib.patches import Patch

    ax.legend(
        handles=[
            Patch(
                facecolor="white",
                edgecolor="orange",
                linewidth=2,
                label="Root molecule",
            )
        ],
        loc="upper left",
        fontsize=9,
    )

    plt.tight_layout()
    return fig

## Run Inference & Visualize

Per molecule: predicted spectrum → mirror plot + similarities (if exp available) → interactive spectrum → DAG.

In [ ]:
for _, row in df_filtered.iterrows():
    smiles = row["standardized_smiles"]
    mol_id = str(row["mol_id"])
    inchikey = row["inchi_key"]
    split_label = row.get("split", "unknown")
    ri = row.get("retention_index", "N/A")
    name = row.get("name", "")

    split_color = {"test": "green", "val": "steelblue", "train": "gray"}.get(
        split_label, "black"
    )

    display(
        HTML(
            f"<hr><h3>{name}</h3>"
            f"<b>SMILES:</b> {smiles} &nbsp;&nbsp; "
            f"<b>mol_id:</b> {mol_id} &nbsp;&nbsp; "
            f"<b>InChIKey:</b> {inchikey}<br>"
            f"<b>Split:</b> <span style='color:{split_color};font-weight:bold'>{split_label}</span> &nbsp;&nbsp; "
            f"<b>Retention index:</b> {ri}"
        )
    )

    # predict
    result = eims_predictor.predict_from_smiles(
        smiles=smiles,
        max_nodes=MAX_NODES,
        threshold=THRESHOLD,
        device=DEVICE,
    )
    pred_spectrum = result["intensities"] / result["intensities"].max()

    # load experimental
    exp_spectrum = load_exp_spectrum(mol_id, n_bins=len(pred_spectrum))

    # ── predicted spectrum ────────────────────────────────────────────────────
    display(HTML("<b>Predicted spectrum</b>"))
    fig_pred = plot_mass_spectrum(
        result["mz_bins"],
        pred_spectrum,
        smiles=smiles,
        fragments=result["fragments"],
        max_fragments=5,
        figsize=(5, 3),
    )
    plt.show()

    # ── mirror plot + similarities ────────────────────────────────────────────
    if exp_spectrum is not None:
        sims = compute_similarities(pred_spectrum, exp_spectrum)
        sim_str = (
            f"cos={sims['cosine']:.3f}  "
            f"entr={sims['entropy']:.3f}  "
            f"wcs={sims['weighted_cosine']:.3f}  "
            f"composite={sims['composite']:.3f}"
        )
        display(
            HTML(
                f"<b>Mirror plot</b> &nbsp; <span style='color:#555'>{sim_str}</span>"
            )
        )
        fig_mirror = plot_mirrored_spectra(
            true_spec=exp_spectrum,
            pred_spec=pred_spectrum,
            true_smiles=smiles,
            figsize=(5, 3),
            title=sim_str,
        )
        plt.show()
    else:
        display(
            HTML(
                "<i style='color:#888'>No experimental spectrum found for this mol_id.</i>"
            )
        )

    # ── interactive spectrum ──────────────────────────────────────────────────
    display(HTML("<b>Interactive spectrum</b>"))
    interactive_spectrum(result, smiles=smiles)

    # ── fragmentation DAG ─────────────────────────────────────────────────────
    display(HTML("<b>Fragmentation DAG</b>"))
    fig_dag = plot_prediction_dag(result)
    if fig_dag is not None:
        plt.show()